In [6]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

raw = pd.read_parquet("data/train-00000-of-00001.parquet")

classification = {
    "appID": "Identification",
    "name": "NonApplicable",
    "release_date": "Applicable",
    "estimated_owners": "Leakage",
    "peak_ccu": "Leakage",
    "required_age": "Applicable",
    "price": "Applicable",
    "dlc_count": "Leakage",
    "detailed_description": "Applicable (difficult)",
    "short_description": "Applicable (difficult)",
    "supported_languages": "Applicable",
    "full_audio_languages": "Applicable",
    "reviews": "Leakage",
    "header_image": "NonApplicable",
    "website": "NonApplicable",
    "support_url": "NonApplicable",
    "support_email": "NonApplicable",
    "windows": "Applicable",
    "mac": "Applicable",
    "linux": "Applicable",
    "metacritic_score": "Leakage",
    "metacritic_url": "NonApplicable",
    "user_score": "Leakage",
    "positive": "Leakage",
    "negative": "Leakage",
    "score_rank": "Leakage",
    "achievements": "NonApplicable",
    "recommendations": "Leakage",
    "notes": "Applicable (difficult)",
    "average_playtime_forever": "Leakage",
    "average_playtime_2weeks": "Leakage",
    "median_playtime_forever": "Leakage",
    "median_playtime_2weeks": "Leakage",
    "developers": "Applicable",
    "publishers": "Applicable",
    "categories": "Applicable",
    "genres": "Applicable",
    "tags": "NonApplicable / Leakage",
    "screenshots": "NonApplicable",
    "movies": "NonApplicable",
    "packages": "NonApplicable",
}

columns_overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "classification": pd.Series(classification),
})
usable_cols = columns_overview.index[
    ~columns_overview["classification"].str.contains("NonApplicable")
]

In [98]:
software_genres = {
    "Game Development", "Design & Illustration", "Animation & Modeling",
    "Utilities", "Video Production", "Software Training", "Audio Production",
    "Photo Editing", "Web Publishing", "Accounting", "Education",
}

def is_software(genres):
    genres = set(genres) if genres is not None else set()
    return len(genres) == 0 or genres <= software_genres

df = raw[usable_cols].copy()
df = df[~df["genres"].apply(is_software)].copy()
df = df[(df["positive"] + df["negative"]) >= 10].copy()

df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["log_positive"] = np.log1p(df["positive"])
df["score"] = df["positive"] / (df["positive"] + df["negative"]) * 100
top = df.groupby("year")["log_positive"].transform("quantile", 0.70)
df["hit"] = ((df["log_positive"] >= top) & (df["score"] > 85)).astype(int)

df["language_count"] = df["supported_languages"].str.len().fillna(0).astype(int)
df["audio_count"] = df["full_audio_languages"].str.len().fillna(0).astype(int)
df["self_published"] = df.apply(
    lambda r: len(set(r["developers"]) & set(r["publishers"])) > 0, axis=1
)
target = "hit"

In [99]:
df_tmp, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_tmp, test_size=0.25, random_state=42)
len(df_train), len(df_val), len(df_test)

(33565, 11189, 11189)

In [124]:
def as_set(x):
    return set(x) if isinstance(x, (list, tuple, np.ndarray)) else set()

def common_values(col):
    counts = df_train[col].explode().value_counts()
    return counts[counts >= min_games].index

dev_count = df_train["developers"].explode().value_counts()

min_games = 0.02 * len(df_train)
top_genres = common_values("genres")
top_cats = common_values("categories")
top_langs = common_values("supported_languages")
top_audio = common_values("full_audio_languages")

def add_features(d):
    new = {
        "log_price": np.log1p(d["price"]),
        "is_free": (d["price"] == 0).astype(int),
        "log_languages": np.log1p(d["language_count"]),
        "log_audio": np.log1p(d["audio_count"]),
        "dev_game_count": d["developers"].apply(
            lambda devs: max((dev_count.get(x, 0) for x in as_set(devs)), default=0)),
        "mature": (d["required_age"] > 0).astype(int),
    }
    new["log_dev_games"] = np.log1p(new["dev_game_count"])
    for c in ["windows", "mac", "linux", "self_published"]:
        new[c] = d[c].astype(int)

    sets = {col: d[col].apply(as_set) for col in
            ["genres", "categories", "supported_languages", "full_audio_languages"]}
    for prefix, col, values in [("genre_", "genres", top_genres),
                                ("cat_", "categories", top_cats),
                                ("lang_", "supported_languages", top_langs),
                                ("audio_", "full_audio_languages", top_audio)]:
        for v in values:
            new[prefix + v] = sets[col].apply(lambda s: int(v in s))

    new = pd.DataFrame(new, index=d.index)
    return pd.concat([d.drop(columns=new.columns, errors="ignore"), new], axis=1)

df_train, df_val, df_test = (add_features(d) for d in (df_train, df_val, df_test))

genre_cols = ["genre_" + g for g in top_genres]
cat_cols = ["cat_" + c for c in top_cats]
lang_cols = ["lang_" + l for l in top_langs]
audio_cols = ["audio_" + a for a in top_audio]

In [125]:
features = (["mature", "self_published", "log_languages", "mac"]
            + genre_cols + cat_cols + lang_cols + audio_cols)

X_train, y_train = df_train[features], df_train[target]
X_val,   y_val   = df_val[features],   df_val[target]
X_test,  y_test  = df_test[features],  df_test[target]

print("Andel hits (train / val / test):",
      round(y_train.mean(), 3), round(y_val.mean(), 3), round(y_test.mean(), 3))

Andel hits (train / val / test): 0.156 0.153 0.161


Jag jämför logistisk regression med en DummyClassifier som alltid väljer den vanligaste klassen. Precision visar hur många rekommenderade spel som faktiskt blir hits, medan recall visar hur många av alla hits på riktigt hittar. F1 sammanväger dessa mått. Accuracy räcker inte eftersom modellen kan få många rätt genom att alltid förutsäga ”inte hit”.

In [126]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(name)
    print(pd.DataFrame([[tn, fp], [fn, tp]],
                       index=["actual flop", "actual hit"],
                       columns=["predicted flop", "predicted hit"]).to_string())
    print("accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
    print("recall   :", round(recall_score(y_true, y_pred, zero_division=0), 3))
    print("F1       :", round(f1_score(y_true, y_pred, zero_division=0), 3))
    print()

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
report("Baseline (always flop)", y_val, baseline.predict(X_val))

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Logistic regression (threshold 0.5)", y_val, model.predict(X_val))

Baseline (always flop)
             predicted flop  predicted hit
actual flop            9474              0
actual hit             1715              0
accuracy : 0.847
precision: 0.0
recall   : 0.0
F1       : 0.0

Logistic regression (threshold 0.5)
             predicted flop  predicted hit
actual flop            6706           2768
actual hit              626           1089
accuracy : 0.697
precision: 0.282
recall   : 0.635
F1       : 0.391



Jag antar att Pixel Harbor kan finansiera 0,5 procent av spelen som jag upskatta som ett någårlunda rimligt tal spel för en studio att kunna financiera. Ett falskt alarm uppskattas kosta 200 000 euro i förlorad investering, medan en missad hit uppskattas kosta 500 000 euro i utebliven vinst.

In [127]:
FUND_SHARE = 0.005
COST_FALSE_ALARM = 200
COST_MISSED_HIT = 500

In [137]:
proba_val = model.predict_proba(X_val)[:, 1]

def threshold_row(t, y_true, proba):
    y_pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": round(t, 2),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 3),
        "games said yes to": int(y_pred.sum()),
        "share said yes to": round(y_pred.mean(), 3),
        "false alarms": int(fp),
        "missed hits": int(fn),
        "cost (k€)": int(fp * COST_FALSE_ALARM + fn * COST_MISSED_HIT),
    }

thresholds = np.arange(0.1, 0.99, 0.01)
table = pd.DataFrame([threshold_row(t, y_val, proba_val) for t in thresholds])
print(table.to_string(index=False))

 threshold  precision  recall    F1  games said yes to  share said yes to  false alarms  missed hits  cost (k€)
      0.10      0.154   0.998 0.266              11136              0.995          9424            3    1886300
      0.11      0.154   0.998 0.267              11111              0.993          9399            3    1881300
      0.12      0.154   0.997 0.267              11088              0.991          9378            5    1878100
      0.13      0.155   0.997 0.268              11057              0.988          9347            5    1871900
      0.14      0.155   0.995 0.268              11010              0.984          9304            9    1865300
      0.15      0.156   0.994 0.269              10957              0.979          9253           11    1856100
      0.16      0.157   0.992 0.270              10875              0.972          9173           13    1841100
      0.17      0.157   0.988 0.272              10758              0.961          9064           21    

In [138]:
t_f1 = table.loc[table["F1"].idxmax(), "threshold"]
t_cost = table.loc[table["cost (k€)"].idxmin(), "threshold"]
t_budget = np.ceil(np.quantile(proba_val, 1 - FUND_SHARE) * 100) / 100   # says yes to at most FUND_SHARE of the games

compare = pd.DataFrame([threshold_row(t, y_val, proba_val) for t in (t_f1, t_cost, t_budget)]).set_index("threshold")
compare.index = [f"highest F1 ({t_f1})", f"lowest cost ({t_cost})", f"fits the budget ({t_budget})"]
print(compare.to_string())

                        precision  recall     F1  games said yes to  share said yes to  false alarms  missed hits  cost (k€)
highest F1 (0.58)           0.327   0.518  0.401               2719              0.243          1831          827     779700
lowest cost (0.73)          0.430   0.289  0.345               1152              0.103           657         1220     741400
fits the budget (0.93)      0.634   0.015  0.030                 41              0.004            15         1689     847500


Jag jämförde endast tröskelvärden där antalet rekommenderade spel ryms inom Pixel Harbors finansieringskapacitet på 0,5 procent av spelen. Budgeten är ett absolut krav, eftersom företaget bara kan finansiera projekt som det har råd med. Jag väljer därför tröskelvärdet 0.93. En högre F1 motiverar inte ett val som överskrider budgeten.

In [140]:
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= THRESHOLD).astype(int)

report(f"TEST: logistic regression, threshold {THRESHOLD}", y_test, pred_test)
print("games said yes to:", int(pred_test.sum()), "of", len(pred_test), f"({pred_test.mean():.1%})")

TEST: logistic regression, threshold 0.93
             predicted flop  predicted hit
actual flop            9372             15
actual hit             1777             25
accuracy : 0.84
precision: 0.625
recall   : 0.014
F1       : 0.027

games said yes to: 40 of 11189 (0.4%)


Jag testade modellen och den valda tröskeln en gång på testdata. Resultatet är ovan.

Appen visar LOVANDE när modellens uppskattade sannolikhet är minst tröskelvärde, annars REKOMMENDERAS INTE.

In [132]:
from IPython.display import HTML

def verdict(p, threshold=THRESHOLD):
    return "LOVANDE" if p >= threshold else "REKOMMENDERAS INTE"

N = 500 

examples = df_test[["positive", "log_positive", "score", "hit"]].head(N).copy()
examples.insert(0, "name", raw.loc[examples.index, "name"])
examples["P(hit)"] = proba_test[:N].round(2)
examples["verdict"] = [verdict(p) for p in proba_test[:N]]
examples = examples.sort_values("positive", ascending=False).head(N)
HTML(f'<div style="max-height:400px; overflow:auto">{examples.round(2).to_html()}</div>')

,name,positive,log_positive,score,hit,P(hit),verdict
78473,Tom Clancy's Rainbow Six® Siege X,312816,12.65,82.97,0,0.90,REKOMMENDERAS INTE
94999,Scrap Mechanic,104512,11.56,92.26,1,0.76,REKOMMENDERAS INTE
41350,Kingdom Come: Deliverance II,96279,11.48,94.08,1,0.97,LOVANDE
68573,XCOM: Enemy Unknown,52717,10.87,93.45,1,0.89,REKOMMENDERAS INTE
120403,Hitman: Absolution™,51769,10.85,92.94,1,0.62,REKOMMENDERAS INTE
80393,Dirty Bomb®,46464,10.75,81.28,0,0.75,REKOMMENDERAS INTE
95941,SILENT HILL 2,41028,10.62,95.31,1,0.87,REKOMMENDERAS INTE
50570,My Singing Monsters,39143,10.58,93.87,1,0.42,REKOMMENDERAS INTE
49488,Fistful of Frags,33130,10.41,91.36,1,0.86,REKOMMENDERAS INTE
59771,BlazBlue Entropy Effect,28794,10.27,95.06,1,0.68,REKOMMENDERAS INTE
